In [ ]:
import subprocess, sys

subprocess.run([
    "uv", "pip", "install", "statsmodels",
    "--python", sys.executable
])

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as f
from pathlib import Path
import pandas as pd
import plotly.graph_objects as go
from dotenv import load_dotenv
import os

load_dotenv()

# ── Spark ────────────────────────────────────────────────────────────
spark = (
    SparkSession.builder.master("local[*]")
    .appName("GenPM-seasonality")
    .config("spark.executor.memory", "24g")
    .config("spark.driver.memory", "16g")
    .config("spark.sql.shuffle.partitions", "200")
    .config("spark.default.parallelism", "8")
    .config("spark.log.level", "ERROR")
    .getOrCreate()
)

# ── Ścieżki ──────────────────────────────────────────────────────────
USER = os.getenv("USER")

prep_data_path = Path(f"/home/{USER}/app/apps/apps/generator/data/shared_dir/preprocessed_dataset")
out_path = Path(f"/home/{USER}/app/apps/apps/generator/data/shared_dir/seasonality")

# ── Dane ─────────────────────────────────────────────────────────────
preprocessed_df = spark.read.parquet(str(prep_data_path / "pm_data_long"))
kpi_definitions_df = spark.read.parquet(str(prep_data_path / "kpi_definitions"))

In [ ]:
preprocessed_df = spark.read.parquet(str(prep_data_path / "pm_data_long"))
kpi_definitions_df = spark.read.parquet(str(prep_data_path / "kpi_definitions"))

preprocessed_df.printSchema()
kpi_definitions_df.printSchema()

print("Liczba wierszy w pm_data_long:", preprocessed_df.count())
print("Liczba unikalnych kpi:", preprocessed_df.select("kpi_id").distinct().count())
print("Liczba unikalnych bts:", preprocessed_df.select("bts_id").distinct().count())

# zakres czasowy
preprocessed_df.agg(
    f.min("start_time").alias("t_min"),
    f.max("start_time").alias("t_max"),
).show(truncate=False)

# rozkład jednostek
kpi_definitions_df.groupBy("unit").count().orderBy(f.desc("count")).show(50, truncate=False)

In [ ]:
# ile distnames per bts?
(
    preprocessed_df
    .select("bts_id", "distname")
    .distinct()
    .groupBy("bts_id")
    .count()
    .orderBy(f.desc("count"))
    .show(20, truncate=False)
)

# weryfikacja: czy dla jednej pary (kpi, bts, start_time) jest wiele distname?
(
    preprocessed_df
    .groupBy("kpi_id", "bts_id", "start_time")
    .count()
    .agg(
        f.min("count").alias("min_per_slot"),
        f.max("count").alias("max_per_slot"),
        f.mean("count").alias("avg_per_slot"),
    )
    .show()
)

# ile unikalnych distname w całym datasecie?
print("Distnames:", preprocessed_df.select("distname").distinct().count())

In [ ]:
# mapping: unit → funkcja agregacji przez komórki
# uzasadnienie:
#   - countery (#, MB) → suma przez komórki BTS
#   - rate-y już znormalizowane (/s, /h, Mbit/s) → średnia (suma byłaby bez sensu)
#   - ratios / poziomy (%, dB, dBm, E, bit/s/Hz) → średnia (nie sumuje się %)
#   - latencje (ms, us, s) → średnia (można zmienic na median dla odporności)

UNIT_AGG = {
    "#":         "sum",
    "MB":        "sum",
    "#/s":       "mean",
    "#/h":       "mean",
    "Mbit/s":    "mean",
    "bit/s/Hz":  "mean",
    "%":         "mean",
    "dB":        "mean",
    "dBm":       "mean",
    "E":         "mean",
    "ms":        "mean",
    "us":        "mean",
    "s":         "mean",
    "m":         "mean",
}

# dołącz unit do danych głównych
df_with_unit = preprocessed_df.join(
    kpi_definitions_df.select("kpi_id", "unit"),
    on="kpi_id",
    how="left",
)

# budujemy wyrażenie agregacji per wiersz przez CASE WHEN per unit
# (Spark nie pozwala na dynamiczne funkcje, ale możemy zbudować to warunkowo)
from pyspark.sql.functions import when

agg_expr = (
    when(f.col("unit").isin([u for u, fn in UNIT_AGG.items() if fn == "sum"]),
         f.sum("kpi_value"))
    .otherwise(f.mean("kpi_value"))
    .alias("kpi_value")
)

df_bts_level = (
    df_with_unit
    .groupBy("kpi_id", "bts_id", "start_time", "unit")
    .agg(
        agg_expr,
        f.count("distname").alias("n_cells"),            # ile komórek weszło do agregatu
        f.count("kpi_value").alias("n_cells_non_null"),  # ile miało wartość != NULL
    )
)

# podgląd
df_bts_level.orderBy("kpi_id", "bts_id", "start_time").show(5, truncate=False)

In [ ]:
# zapis — to jest nowa baza do całej dalszej analizy
bts_level_path = out_path / "pm_data_bts_level"
df_bts_level.write.mode("overwrite").parquet(str(bts_level_path))
df_bts_level = spark.read.parquet(str(bts_level_path))

# weryfikacja: teraz max 1 wiersz per (kpi, bts, start_time)
(
    df_bts_level
    .groupBy("kpi_id", "bts_id", "start_time")
    .count()
    .agg(
        f.min("count").alias("min"),
        f.max("count").alias("max"),
    )
    .show()
)
# oczekiwane: min=1, max=1 ✓

# ile wierszy per para (kpi, bts) teraz?
(
    df_bts_level
    .groupBy("kpi_id", "bts_id")
    .count()
    .agg(
        f.min("count").alias("min_rows"),
        f.max("count").alias("max_rows"),
        f.mean("count").alias("avg_rows"),
    )
    .show()
)
# oczekiwane: min_rows ≥ 1, max_rows ≈ total_hours (2160)

In [ ]:
# np. slot jest "wiarygodny" jeśli ≥50% komórek BTS go zasiliło
df_bts_level = df_bts_level.withColumn(
    "cell_coverage_ratio",
    f.col("n_cells_non_null") / f.col("n_cells"),
)

# jeśli chcesz być ostrożny — zastąp kpi_value przez NULL jeśli za mało komórek
df_bts_level = df_bts_level.withColumn(
    "kpi_value_reliable",
    f.when(f.col("cell_coverage_ratio") >= 0.5, f.col("kpi_value"))
     .otherwise(f.lit(None).cast("double"))
)

In [ ]:
# zakres czasowy (oczekiwana liczba slotów godzinowych)
time_bounds = df_bts_level.agg(
    f.min("start_time").alias("t_min"),
    f.max("start_time").alias("t_max"),
).collect()[0]
t_min, t_max = time_bounds["t_min"], time_bounds["t_max"]
total_hours = int((t_max - t_min).total_seconds() // 3600) + 1
print(f"Zakres: {t_min} → {t_max}, oczekiwana liczba godzin: {total_hours}")

stats_df = (
    df_bts_level
    .groupBy("kpi_id", "bts_id", "unit")
    .agg(
        f.count("*").alias("n_rows"),
        f.count("kpi_value_reliable").alias("n_non_null"),
        f.min("kpi_value_reliable").alias("v_min"),
        f.max("kpi_value_reliable").alias("v_max"),
        f.stddev("kpi_value_reliable").alias("v_std"),
        f.mean("kpi_value_reliable").alias("v_mean"),
        f.expr("percentile_approx(kpi_value_reliable, 0.5)").alias("v_median"),
        f.expr("percentile_approx(kpi_value_reliable, 0.99)").alias("v_p99"),
    )
    .withColumn("coverage", f.col("n_non_null") / f.lit(total_hours))
    .withColumn(
        "is_constant",
        (f.col("v_min") == f.col("v_max")) | (f.col("v_std").isNull()) | (f.col("v_std") < 1e-12),
    )
    .withColumn(
        "is_analyzable",
        (~f.col("is_constant"))
        & (f.col("coverage") >= 0.5)
        & (f.col("n_non_null") >= 24 * 14)
    )
)

stats_path = out_path / "series_stats"
stats_df.write.mode("overwrite").parquet(str(stats_path))
stats_df = spark.read.parquet(str(stats_path))

total = stats_df.count()
analyzable = stats_df.filter(f.col("is_analyzable")).count()
constant = stats_df.filter(f.col("is_constant")).count()
print(f"Wszystkich par (kpi, bts): {total}")
print(f"Stałych:                   {constant}")
print(f"Analizowalnych:            {analyzable} ({100*analyzable/total:.1f}%)")

# rozkład per unit
stats_df.filter(f.col("is_analyzable")).groupBy("unit").count().orderBy(f.desc("count")).show(50, False)

In [ ]:
# Poprawione
stats_df = (
    df_bts_level
    .groupBy("kpi_id", "bts_id", "unit")
    .agg(
        f.count("*").alias("n_rows"),
        f.count("kpi_value_reliable").alias("n_non_null"),
        f.min("kpi_value_reliable").alias("v_min"),
        f.max("kpi_value_reliable").alias("v_max"),
        f.stddev("kpi_value_reliable").alias("v_std"),
        f.mean("kpi_value_reliable").alias("v_mean"),
        f.expr("percentile_approx(kpi_value_reliable, 0.01)").alias("v_p01"),
        f.expr("percentile_approx(kpi_value_reliable, 0.5)").alias("v_median"),
        f.expr("percentile_approx(kpi_value_reliable, 0.99)").alias("v_p99"),
        f.expr("percentile_approx(kpi_value_reliable, 0.25)").alias("v_p25"),
        f.expr("percentile_approx(kpi_value_reliable, 0.75)").alias("v_p75"),
    )
    .withColumn("coverage", f.col("n_non_null") / f.lit(total_hours))
    # IQR znormalizowany — robust miara "ile się to rusza"
    .withColumn(
        "relative_iqr",
        (f.col("v_p75") - f.col("v_p25")) / f.greatest(f.abs(f.col("v_median")), f.lit(1.0))
    )
    # spread 1–99 percentyl (do diagnostyki)
    .withColumn(
        "relative_p1_99",
        (f.col("v_p99") - f.col("v_p01")) / f.greatest(f.abs(f.col("v_median")), f.lit(1.0))
    )
    .withColumn(
        "is_constant",
        (f.col("v_min") == f.col("v_max")) | (f.col("v_std").isNull()) | (f.col("v_std") < 1e-12),
    )
    # NOWE: saturacja przy 100 (success rate) lub przy 0 (failure rate)
    .withColumn(
        "is_saturated",
        (f.col("unit") == "%") & (
            ((f.col("v_median") >= 99.0) & (f.col("v_p25") >= 99.0)) |  # stuck near 100
            ((f.col("v_median") <= 0.5)  & (f.col("v_p75") <= 1.0))     # stuck near 0
        )
    )
    # NOWE: za mała zmienność (IQR < 1% mediany) — szum, nie sygnał
    .withColumn(
        "is_low_variability",
        f.col("relative_iqr") < 0.01
    )
    .withColumn(
        "is_analyzable",
        (~f.col("is_constant"))
        & (~f.col("is_saturated"))
        & (~f.col("is_low_variability"))
        & (f.col("coverage") >= 0.5)
        & (f.col("n_non_null") >= 24 * 14)
    )
)

# zapisz na nowo
stats_df.write.mode("overwrite").parquet(str(stats_path))
stats_df = spark.read.parquet(str(stats_path))

# rozkład powodów wykluczenia
stats_df.groupBy("is_constant", "is_saturated", "is_low_variability", "is_analyzable").count().show()

# ile par odpada przez nowe filtry
print(f"Skonstantowane:     {stats_df.filter(f.col('is_constant')).count()}")
print(f"Saturowane:         {stats_df.filter(f.col('is_saturated') & ~f.col('is_constant')).count()}")
print(f"Niska zmienność:    {stats_df.filter(f.col('is_low_variability') & ~f.col('is_constant') & ~f.col('is_saturated')).count()}")
print(f"Analizowalne:       {stats_df.filter(f.col('is_analyzable')).count()}")

In [ ]:
analyzable_pairs = stats_df.filter(f.col("is_analyzable")).select("kpi_id", "bts_id", "unit")
print(f"Liczba par do analizy: {analyzable_pairs.count()}")

# pełna siatka godzin
grid = spark.sql(f"""
    SELECT explode(sequence(
        TIMESTAMP '{t_min}',
        TIMESTAMP '{t_max}',
        INTERVAL 1 HOUR
    )) AS start_time
""")

full_grid = analyzable_pairs.crossJoin(grid)
print(f"Oczekiwana liczba wierszy: {analyzable_pairs.count() * total_hours}")

df_aligned = (
    full_grid
    .join(
        df_bts_level.select("kpi_id", "bts_id", "start_time", "kpi_value_reliable",
                            "n_cells", "n_cells_non_null", "cell_coverage_ratio"),
        on=["kpi_id", "bts_id", "start_time"],
        how="left",
    )
)

aligned_path = out_path / "pm_data_aligned"
df_aligned.write.mode("overwrite").parquet(str(aligned_path))
df_aligned = spark.read.parquet(str(aligned_path))

# weryfikacja
(
    df_aligned
    .groupBy("kpi_id", "bts_id")
    .count()
    .agg(f.min("count").alias("min_rows"), f.max("count").alias("max_rows"))
    .show()
)
# oczekiwane: min == max == total_hours

In [ ]:
sample_pair = (
    stats_df
    .filter(f.col("is_analyzable") & (f.col("coverage") > 0.95))
    .orderBy(f.desc("v_std"))
    .limit(1)
    .collect()[0]
)
sample_kpi, sample_bts = sample_pair["kpi_id"], sample_pair["bts_id"]
print(f"Sample: kpi={sample_kpi}, bts={sample_bts}, unit={sample_pair['unit']}")

sample_pdf = (
    df_aligned
    .filter((f.col("kpi_id") == sample_kpi) & (f.col("bts_id") == sample_bts))
    .orderBy("start_time")
    .toPandas()
)

# statystyki — wykryj outliery
y = sample_pdf["kpi_value_reliable"].astype(float)
print(y.describe(percentiles=[0.01, 0.5, 0.99, 0.999]))
print(f"max / median = {y.max() / max(y.median(), 1e-9):.1f}x")

# wykres
fig = go.Figure()
fig.add_trace(go.Scatter(x=sample_pdf["start_time"], y=y, mode="lines"))
fig.update_layout(title=f"Surowy szereg: {sample_kpi} @ {sample_bts}", height=400)
fig.show()

In [ ]:
from statsmodels.tsa.seasonal import MSTL
from statsmodels.tsa.stattools import acf
import numpy as np
from pyspark.sql import types as t

def analyze_one_series(pdf: pd.DataFrame) -> pd.DataFrame:
    kpi_id = pdf["kpi_id"].iloc[0]
    bts_id = pdf["bts_id"].iloc[0]

    pdf = pdf.sort_values("start_time").set_index("start_time")
    y = pdf["kpi_value_reliable"].astype(float)

    result = {
        "kpi_id": kpi_id, "bts_id": bts_id,
        "n_obs": int(y.notna().sum()),
        "n_outliers": 0,
        "acf_24": np.nan, "acf_168": np.nan,
        "strength_daily": np.nan, "strength_weekly": np.nan,
        "fft_peak_period_h": np.nan, "fft_peak_power_share": np.nan,
        "error": None,
        "outliers_applied": False,
    }

    if result["n_obs"] < 24 * 14:
        result["error"] = "too_few_obs"
        return pd.DataFrame([result])

    try:
        # --- MAD outlier removal ---
        med = y.median()
        mad = (y - med).abs().median()

        if mad > 1e-12:
            z_robust = 0.6745 * (y - med).abs() / mad
            outlier_mask = z_robust > 7  # luźniej (z 5 do 7) — łapie tylko prawdziwe spike'i

            # cap: jeśli wykryto > 3% punktów jako outliery, nie ufamy MAD-owi
            n_potential = int(outlier_mask.sum())
            if n_potential > 0.03 * y.notna().sum():
                result["n_outliers"] = n_potential
                result["outliers_applied"] = False  # zaznaczamy że nie wycięliśmy
            else:
                result["n_outliers"] = n_potential
                result["outliers_applied"] = True
                y = y.mask(outlier_mask)
        else:
            result["n_outliers"] = 0
            result["outliers_applied"] = False

        # --- imputacja ---
        y_filled = y.interpolate(method="linear", limit=3, limit_direction="both")
        y_filled = y_filled.fillna(y_filled.median())

        if y_filled.std() < 1e-12:
            result["error"] = "constant_after_cleaning"
            return pd.DataFrame([result])

        # --- ACF ---
        max_lag = min(200, len(y_filled) - 1)
        acf_vals = acf(y_filled, nlags=max_lag, fft=True, missing="conservative")
        if max_lag >= 24:  result["acf_24"]  = float(acf_vals[24])
        if max_lag >= 168: result["acf_168"] = float(acf_vals[168])

        # --- MSTL ---
        if len(y_filled) >= 168 * 2:
            mstl = MSTL(y_filled, periods=(24, 168)).fit()
            resid = mstl.resid
            seas_daily  = mstl.seasonal.iloc[:, 0]
            seas_weekly = mstl.seasonal.iloc[:, 1]
            var_resid     = np.nanvar(resid)
            var_rs_daily  = np.nanvar(resid + seas_daily)
            var_rs_weekly = np.nanvar(resid + seas_weekly)
            if var_rs_daily  > 1e-12: result["strength_daily"]  = float(max(0, 1 - var_resid / var_rs_daily))
            if var_rs_weekly > 1e-12: result["strength_weekly"] = float(max(0, 1 - var_resid / var_rs_weekly))

        # --- FFT (znormalizowana siła piku) ---
        y_centered = y_filled - y_filled.mean()
        fft_vals = np.fft.rfft(y_centered.values)
        fft_freqs = np.fft.rfftfreq(len(y_centered), d=1.0)
        power = np.abs(fft_vals) ** 2
        if len(power) > 1 and power[1:].sum() > 0:
            idx = np.argmax(power[1:]) + 1
            if fft_freqs[idx] > 0:
                result["fft_peak_period_h"] = float(1.0 / fft_freqs[idx])
                # share = jaki % całkowitej mocy (ex-DC) idzie do dominującego piku
                result["fft_peak_power_share"] = float(power[idx] / power[1:].sum())

    except Exception as e:
        result["error"] = f"{type(e).__name__}: {str(e)[:200]}"

    return pd.DataFrame([result])


result_schema = t.StructType([
    t.StructField("kpi_id",                t.StringType()),
    t.StructField("bts_id",                t.StringType()),
    t.StructField("n_obs",                 t.IntegerType()),
    t.StructField("n_outliers",            t.IntegerType()),
    t.StructField("acf_24",                t.DoubleType()),
    t.StructField("acf_168",               t.DoubleType()),
    t.StructField("strength_daily",        t.DoubleType()),
    t.StructField("strength_weekly",       t.DoubleType()),
    t.StructField("fft_peak_period_h",     t.DoubleType()),
    t.StructField("fft_peak_power_share",  t.DoubleType()),
    t.StructField("error",                 t.StringType()),
    t.StructField("outliers_applied",      t.BooleanType()),
])

In [ ]:
# test pojedynczy
test_pdf = (
    df_aligned
    .filter((f.col("kpi_id") == sample_kpi) & (f.col("bts_id") == sample_bts))
    .toPandas()
)
print(analyze_one_series(test_pdf))

In [ ]:
# sample 50 par
sample_pairs = (
    stats_df.filter(f.col("is_analyzable"))
    .orderBy(f.desc("v_std"))
    .limit(50)
    .select("kpi_id", "bts_id")
)

sample_data = df_aligned.join(sample_pairs, on=["kpi_id", "bts_id"], how="inner")

sample_results = (
    sample_data
    .groupBy("kpi_id", "bts_id")
    .applyInPandas(analyze_one_series, schema=result_schema)
)

sample_pdf = sample_results.toPandas()
print(sample_pdf.shape)
print("\nBłędy:")
print(sample_pdf["error"].value_counts(dropna=False))
print("\nStatystyki sezonowości:")
print(sample_pdf[["strength_daily", "strength_weekly", "fft_peak_period_h", "n_outliers"]].describe())

In [ ]:
sample_pairs_random = (
    stats_df.filter(f.col("is_analyzable"))
    .orderBy(f.rand(seed=42))
    .limit(100)
    .select("kpi_id", "bts_id")
)

sample_data_random = df_aligned.join(sample_pairs_random, on=["kpi_id", "bts_id"], how="inner")
sample_results_random = (
    sample_data_random
    .groupBy("kpi_id", "bts_id")
    .applyInPandas(analyze_one_series, schema=result_schema)
)

random_pdf = sample_results_random.toPandas()
print(random_pdf[["strength_daily", "strength_weekly", "n_outliers"]].describe(
    percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]
))

# histogram
fig = go.Figure()
fig.add_trace(go.Histogram(x=random_pdf["strength_daily"], name="daily", opacity=0.6, nbinsx=30))
fig.add_trace(go.Histogram(x=random_pdf["strength_weekly"], name="weekly", opacity=0.6, nbinsx=30))
fig.update_layout(barmode="overlay", title="Rozkład siły sezonowości (random sample)")
fig.show()

In [ ]:
# zobacz problematyczne przypadki
hi_outliers = random_pdf[random_pdf["n_outliers"] >= 100].sort_values("n_outliers", ascending=False)
print(f"Szeregów z ≥100 outlierów: {len(hi_outliers)}")
print(hi_outliers[["kpi_id", "bts_id", "n_obs", "n_outliers", "strength_daily", "strength_weekly"]].head(10))

# dołącz metadane — jakie KPI to są?
hi_pairs_df = spark.createDataFrame(hi_outliers[["kpi_id", "bts_id"]])
(
    hi_pairs_df
    .join(kpi_definitions_df, on="kpi_id", how="left")
    .join(stats_df.select("kpi_id", "bts_id", "unit", "v_median", "v_p99", "v_max"), 
          on=["kpi_id", "bts_id"], how="left")
    .show(20, truncate=False)
)

In [ ]:
spark.conf.set("spark.sql.shuffle.partitions", "400")

In [ ]:
analyzable_pairs_only = stats_df.filter(f.col("is_analyzable")).select("kpi_id", "bts_id")
full_data = df_aligned.join(analyzable_pairs_only, on=["kpi_id", "bts_id"], how="inner")

seasonality_results = (
    full_data
    .groupBy("kpi_id", "bts_id")
    .applyInPandas(analyze_one_series, schema=result_schema)
)

results_path = out_path / "seasonality_results"
seasonality_results.write.mode("overwrite").parquet(str(results_path))
seasonality_results = spark.read.parquet(str(results_path))

print("Liczba przeanalizowanych par:", seasonality_results.count())
seasonality_results.filter(f.col("error").isNotNull()).groupBy("error").count().show(truncate=False)

In [ ]:
DAILY_THR = 0.5
WEEKLY_THR = 0.3

results_classified = (
    seasonality_results
    .join(kpi_definitions_df, on="kpi_id", how="left")
    .withColumn("has_daily",  f.col("strength_daily")  >= DAILY_THR)
    .withColumn("has_weekly", f.col("strength_weekly") >= WEEKLY_THR)
    .withColumn(
        "seasonality_class",
        f.when(f.col("has_daily") & f.col("has_weekly"), "both")
         .when(f.col("has_daily"),  "only_daily")
         .when(f.col("has_weekly"), "only_weekly")
         .otherwise("none")
    )
)

results_classified.groupBy("seasonality_class").count().orderBy(f.desc("count")).show()

# rozkład klas per unit
results_classified.groupBy("unit", "seasonality_class").count() \
    .orderBy("unit", f.desc("count")).show(100, False)

# zapis
final_path = out_path / "seasonality_final"
results_classified.write.mode("overwrite").parquet(str(final_path))

In [ ]:
import plotly.express as px


def plot_weekly_heatmap(kpi_id_v: str, bts_id_v: str):
    pdf = (
        df_aligned
        .filter((f.col("kpi_id") == kpi_id_v) & (f.col("bts_id") == bts_id_v))
        .withColumn("dow",  f.dayofweek("start_time"))   # 1=Sun ... 7=Sat
        .withColumn("hour", f.hour("start_time"))
        .groupBy("dow", "hour")
        .agg(f.mean("kpi_value_reliable").alias("mean_val"))
        .toPandas()
    )
    pivot = pdf.pivot(index="dow", columns="hour", values="mean_val")
    pivot = pivot.reindex([2,3,4,5,6,7,1])
    pivot.index = ["Pon","Wt","Śr","Cz","Pt","Sob","Nd"]

    fig = px.imshow(pivot, aspect="auto",
                    labels=dict(x="Godzina", y="Dzień tyg.", color="Średnia"),
                    title=f"Heatmapa: {kpi_id_v} @ {bts_id_v}",
                    color_continuous_scale="Viridis")
    fig.show()


def plot_mstl(kpi_id_v: str, bts_id_v: str):
    pdf = (
        df_aligned
        .filter((f.col("kpi_id") == kpi_id_v) & (f.col("bts_id") == bts_id_v))
        .orderBy("start_time").toPandas().set_index("start_time")
    )
    y = pdf["kpi_value_reliable"].astype(float)
    med = y.median()
    mad = (y - med).abs().median()
    if mad > 1e-12:
        y = y.mask(0.6745 * (y - med).abs() / mad > 5)
    y_f = y.interpolate(method="linear", limit=3, limit_direction="both").fillna(y.median())
    mstl = MSTL(y_f, periods=(24, 168)).fit()

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=y_f.index, y=y_f.values, name="obserwowany"))
    fig.add_trace(go.Scatter(x=y_f.index, y=mstl.trend, name="trend"))
    fig.add_trace(go.Scatter(x=y_f.index, y=mstl.seasonal.iloc[:, 0], name="sez. 24h"))
    fig.add_trace(go.Scatter(x=y_f.index, y=mstl.seasonal.iloc[:, 1], name="sez. 168h"))
    fig.update_layout(title=f"MSTL: {kpi_id_v} @ {bts_id_v}", height=600)
    fig.show()


# top 3 pary z najsilniejszą sezonowością tygodniową
top = (
    results_classified
    .filter(f.col("seasonality_class") == "both")
    .orderBy(f.desc("strength_weekly"))
    .limit(3)
    .select("kpi_id", "bts_id").collect()
)
for row in top:
    plot_weekly_heatmap(row["kpi_id"], row["bts_id"])
    plot_mstl(row["kpi_id"], row["bts_id"])

# top 3 pary z najsilniejszą sezonowością tygodniową
top = (
    results_classified
    .filter(f.col("seasonality_class") == "both")
    .orderBy(f.desc("strength_daily"))
    .limit(3)
    .select("kpi_id", "bts_id").collect()
)
for row in top:
    plot_weekly_heatmap(row["kpi_id"], row["bts_id"])
    plot_mstl(row["kpi_id"], row["bts_id"])